In [4]:
import tiktoken
from torch.cuda import temperature

In [5]:
tokenizer = tiktoken.encoding_for_model("gpt-4o")
text = "안녕하세요, AI 에이전트 실습 중입니다!"

token_ids = tokenizer.encode(text)
print(text)
print(len(token_ids))
print(token_ids)
print("-----------------------------")


안녕하세요, AI 에이전트 실습 중입니다!
13
[14307, 171731, 11, 20837, 47061, 2186, 9516, 7984, 27365, 8662, 19078, 27001, 0]
-----------------------------


In [6]:
for t_id in token_ids:
    token_str = tokenizer.decode([t_id])
    print(f"{t_id}: {token_str}")
print()

14307: 안
171731: 녕하세요
11: ,
20837:  AI
47061:  에
2186: 이
9516: 전
7984: 트
27365:  실
8662: 습
19078:  중
27001: 입니다
0: !



In [7]:
decoded_text = tokenizer.decode(token_ids)
print(f"{decoded_text}")

안녕하세요, AI 에이전트 실습 중입니다!


In [8]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
reviews = [
    "배송도 빠르고 품질도 최고에요!",
    "별로네요, 다시는 안 살 것 같아요.",
    "그냥 그래요. 가격은 괜찮은데 좀 아쉬움"
]

for r in reviews:
    tokens = tok.tokenize(r)
    print(f"{r}: {tokens}")
    print(len(tokens))

배송도 빠르고 품질도 최고에요!: ['배', '##송', '##도', '빠', '##르고', '품', '##질', '##도', '최고', '##에', '##요', '!']
12
별로네요, 다시는 안 살 것 같아요.: ['별', '##로', '##네', '##요', ',', '다시', '##는', '안', '살', '것', '같', '##아', '##요', '.']
14
그냥 그래요. 가격은 괜찮은데 좀 아쉬움: ['그', '##냥', '그', '##래', '##요', '.', '가', '##격', '##은', '괜', '##찮', '##은', '##데', '좀', '아', '##쉬', '##움']
17


In [18]:
from transformers import pipeline
classifier = pipeline("sentiment-analysis")

result = classifier("별로임")
print(result)

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.7954825162887573}]


In [1]:
from pathlib import Path
import os
print(os.getcwd())
print(Path.cwd())


cwd = Path.cwd()

for p in [cwd, *cwd.parents]:
    env_path = p / ".env"
    print(env_path, env_path.exists())

/mnt/c/Users/sdh08/PycharmProjects/PythonProject1/개인
/mnt/c/Users/sdh08/PycharmProjects/PythonProject1/개인
/mnt/c/Users/sdh08/PycharmProjects/PythonProject1/개인/.env False
/mnt/c/Users/sdh08/PycharmProjects/PythonProject1/.env True
/mnt/c/Users/sdh08/PycharmProjects/.env False
/mnt/c/Users/sdh08/.env False
/mnt/c/Users/.env False
/mnt/c/.env False
/mnt/.env False
/.env False


In [9]:
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv, find_dotenv

load_dotenv()
client = OpenAI()

def embed(text, dim = 1536):
    res = client.embeddings.create(
        input = text,
        model = "text-embedding-3-small",
        dimensions=dim
    )
    return np.array(res.data[0].embedding)

def cosine_similarity(a,b):
    return np.dot(a,b) / (np.linalg.norm(a) * np.linalg.norm(b))

query = "강아지가 침대에서 낮잠을 잔다"
docs = [
    '고양이가 침대에서 자고 있다.',
    '강아지가 침대에서 자고있다',
    '강아지가 자고있다.'
]

def rank(query, docs, dim=1536):
    q = embed(query, dim)
    scored = [(doc, cosine_similarity(q, embed(doc,dim))) for doc in docs]
    return sorted(scored, key=lambda x: -x[1])

for doc, score in rank(query, docs):
    print(f"{doc}: {score}")

강아지가 침대에서 자고있다: 0.8892276239375144
강아지가 자고있다.: 0.6735558363489476
고양이가 침대에서 자고 있다.: 0.5426399567768116


In [10]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages = [
        {"role":"system", "content" : "너는 한국어 번역가다. 영어를 한국어로만 번역해라"},
        {"role":"user","content": "the weather is nice"}
    ],
    temperature = 0.7,
    max_tokens=500
)
print(response.choices[0].message.content)


날씨가 좋다.
